In [ ]:
from sklearn.ensemble import RandomForestClassifier # Import the missing class
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import r2_score, mean_squared_error
import numpy as np
import pandas as pd
import joblib

In [ ]:
#Feature generation whose pvalue<0.5 and Correlatoin>0.1

import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from scipy.stats import pearsonr

# Load data
df = pd.read_csv('features.csv')

# Apply log transformation to the label
df['label'] = np.log1p(df['label'])  # Use log1p to safely handle zeros

# Separate features and label
features = [col for col in df.columns if col != 'label']
X = df[features]
y = df['label']

# Standardize the feature columns
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Put scaled features back into a DataFrame
X_scaled_df = pd.DataFrame(X_scaled, columns=features)

# Pearson correlation analysis between each feature and the (transformed) label
correlations = []
p_values = []

for feature in features:
    corr, p = pearsonr(X_scaled_df[feature], y)
    correlations.append(corr)
    p_values.append(p)

# Create a DataFrame with correlation results
result_df = pd.DataFrame({
    'Feature': features,
    'Correlation': correlations,
    'P-Value': p_values
})

# Select features based on correlation strength and statistical significance
selected_features = result_df[(result_df['P-Value'] < 0.5) & (result_df['Correlation'].abs() > 0.3)]

# Display selected features
print("Selected features (|correlation| > 0.2 and p < 0.5):")
print(selected_features)

# Get list of selected feature names
selected_columns = selected_features['Feature'].tolist()

# Final filtered DataFrame (scaled features + log-transformed label)
df_filtered = X_scaled_df[selected_columns]
df_filtered['label'] = y  # Add back the log-transformed label

# Save to CSV
df_filtered.to_csv('selected_features.csv', index=False)
print("Saved selected and preprocessed features to 'selected_features.csv'")

✅ Selected features (|correlation| > 0.2 and p < 0.5):
    Feature  Correlation       P-Value
425   BTC_T     0.328353  1.083972e-27
426   BTC_H     0.327475  1.521895e-27
427   BTC_S     0.328159  1.168573e-27
428   BTC_D     0.329940  5.856248e-28
434   RRI_G     0.327278  1.641444e-27
435   RRI_H     0.315289  1.502759e-25
443   RRI_R     0.336949  3.694213e-29
447   RRI_W     0.449724  3.640042e-53
448   RRI_Y     0.324052  5.647111e-27
451   DDR_D     0.377264  1.097753e-36
452   DDR_E     0.314586  1.946083e-25
453   DDR_F     0.335135  7.605188e-29
454   DDR_G     0.377101  1.183664e-36
455   DDR_H     0.368225  6.660189e-35
456   DDR_I     0.339687  1.230864e-29
457   DDR_K     0.341753  5.333395e-30
459   DDR_M     0.309145  1.406306e-24
460   DDR_N     0.313342  3.070325e-25
461   DDR_P     0.316330  1.023534e-25
462   DDR_Q     0.355404  1.805163e-32
463   DDR_R     0.344936  1.451175e-30
464   DDR_S     0.351910  7.950118e-32
465   DDR_T     0.367960  7.497770e-35
467   DDR

/tmp/ipython-input-2566134859.py:54: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_filtered['label'] = y  # Add back the log-transformed label


In [ ]:

import pandas as pd
from sklearn.model_selection import train_test_split
import numpy as np


X = df_filtered.drop('label', axis=1)
y = df_filtered['label']

# Try with 5 bins (fewer bins to avoid empty or single-sample bins)
num_bins = 5
y_binned = pd.cut(y, bins=num_bins, labels=False)

# Check bin counts (optional debug)
print(pd.Series(y_binned).value_counts())

# Split with stratification
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y_binned
)

label
0    376
2    303
3    213
1    126
4     27
Name: count, dtype: int64


In [ ]:
X_train.to_csv('X_train.csv', index=False)
y_train.to_csv('y_train.csv', index=False)
X_test.to_csv('X_test.csv', index=False)
y_test.to_csv('y_test.csv', index=False)

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.svm import SVR
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, AdaBoostRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.preprocessing import StandardScaler
import xgboost as xgb
import lightgbm as lgb
from scipy.stats import pearsonr

# Load preprocessed dataset
df = pd.read_csv('selected_features.csv')

# Features and label
X = df.drop('label', axis=1)
y_log = df['label']  # already log-transformed

# Standardize label
y_scaler = StandardScaler()
y_scaled = y_scaler.fit_transform(y_log.values.reshape(-1, 1)).flatten()

# Stratified binning for split
num_bins = 5
y_binned = pd.cut(y_scaled, bins=num_bins, labels=False)

# Split with stratification
X_train, X_test, y_train, y_test = train_test_split(
    X, y_scaled, test_size=0.2, random_state=42, stratify=y_binned
)

# Define models
models = {
    "SVR": SVR(),
    "Linear Regression": LinearRegression(),
    "Ridge": Ridge(),
    "Lasso": Lasso(),
    "Random Forest": RandomForestRegressor(),
    "Gradient Boosting": GradientBoostingRegressor(),
    "AdaBoost": AdaBoostRegressor(),
    "K-Nearest Neighbors": KNeighborsRegressor(),
    "XGBoost": xgb.XGBRegressor(),
    "LightGBM": lgb.LGBMRegressor()
}

# Parameter grid
param_grid = {
    'SVR': {'C': [0.1, 1, 10], 'epsilon': [0.01, 0.1], 'gamma': ['scale', 'auto']},
    'Random Forest': {'n_estimators': [50, 100, 200], 'max_depth': [None, 10, 20]},
    'XGBoost': {'n_estimators': [50, 100, 200], 'learning_rate': [0.01, 0.1, 0.2]},
    'LightGBM': {'n_estimators': [50, 100, 200], 'learning_rate': [0.01, 0.1, 0.2]}
}

# Helper function to safely apply inverse_transform and expm1
def safe_inverse_expm1(y_scaled, scaler, clip_max=100):
    y_scaled = np.nan_to_num(y_scaled, nan=0.0, posinf=1e6, neginf=-1e6)
    y_unscaled = scaler.inverse_transform(y_scaled.reshape(-1, 1)).flatten()
    y_unscaled = np.clip(y_unscaled, a_min=None, a_max=clip_max)
    return np.expm1(y_unscaled)

# Evaluation function
def evaluate_models(models, param_grid, X_train, y_train, X_test, y_test):
    results = []
    kf = KFold(n_splits=5, shuffle=True, random_state=42)

    for model_name, model in models.items():
        print(f"\nTraining {model_name}...")

        grid_search = GridSearchCV(
            estimator=model,
            param_grid=param_grid.get(model_name, {}),
            cv=kf,
            scoring='neg_mean_squared_error',
            return_train_score=True
        )
        grid_search.fit(X_train, y_train)

        best_model = grid_search.best_estimator_
        print(f"Best parameters for {model_name}: {grid_search.best_params_}")

        train_metrics = {'mse': [], 'mae': [], 'rmse': [], 'pcc': [], 'r2': []}
        val_metrics = {'mse': [], 'mae': [], 'rmse': [], 'pcc': [], 'r2': []}

        for train_idx, val_idx in kf.split(X_train):
            X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
            y_tr, y_val = y_train[train_idx], y_train[val_idx]

            best_model.fit(X_tr, y_tr)

            y_tr_pred_scaled = best_model.predict(X_tr)
            y_val_pred_scaled = best_model.predict(X_val)

            y_tr_pred = safe_inverse_expm1(y_tr_pred_scaled, y_scaler)
            y_val_pred = safe_inverse_expm1(y_val_pred_scaled, y_scaler)
            y_tr_orig = safe_inverse_expm1(y_tr, y_scaler)
            y_val_orig = safe_inverse_expm1(y_val, y_scaler)

            train_metrics['mse'].append(mean_squared_error(y_tr_orig, y_tr_pred))
            train_metrics['mae'].append(mean_absolute_error(y_tr_orig, y_tr_pred))
            train_metrics['rmse'].append(np.sqrt(mean_squared_error(y_tr_orig, y_tr_pred)))
            train_metrics['r2'].append(r2_score(y_tr_orig, y_tr_pred))
            train_metrics['pcc'].append(pearsonr(y_tr_orig, y_tr_pred)[0])

            val_metrics['mse'].append(mean_squared_error(y_val_orig, y_val_pred))
            val_metrics['mae'].append(mean_absolute_error(y_val_orig, y_val_pred))
            val_metrics['rmse'].append(np.sqrt(mean_squared_error(y_val_orig, y_val_pred)))
            val_metrics['r2'].append(r2_score(y_val_orig, y_val_pred))
            val_metrics['pcc'].append(pearsonr(y_val_orig, y_val_pred)[0])

        best_model.fit(X_train, y_train)
        y_test_pred_scaled = best_model.predict(X_test)

        y_test_pred = safe_inverse_expm1(y_test_pred_scaled, y_scaler)
        y_test_orig = safe_inverse_expm1(y_test, y_scaler)

        test_mse = mean_squared_error(y_test_orig, y_test_pred)
        test_mae = mean_absolute_error(y_test_orig, y_test_pred)
        test_rmse = np.sqrt(test_mse)
        test_r2 = r2_score(y_test_orig, y_test_pred)
        test_pcc = pearsonr(y_test_orig, y_test_pred)[0]

        result = {
            'Regressor': model_name,
            'Training MSE': np.mean(train_metrics['mse']),
            'Training MAE': np.mean(train_metrics['mae']),
            'Training RMSE': np.mean(train_metrics['rmse']),
            'Training PCC': np.mean(train_metrics['pcc']),
            'Training R2': np.mean(train_metrics['r2']),
            'Validation MSE': test_mse,
            'Validation MAE': test_mae,
            'Validation RMSE': test_rmse,
            'Validation PCC': test_pcc,
            'Validation R2': test_r2
        }
        results.append(result)

    return pd.DataFrame(results)

# Run evaluation
performance_results = evaluate_models(models, param_grid, X_train, y_train, X_test, y_test)

# Format and export results
formatted_results = performance_results[[
    'Regressor',
    'Training MSE', 'Training MAE', 'Training RMSE', 'Training PCC', 'Training R2',
    'Validation MSE', 'Validation MAE', 'Validation RMSE', 'Validation PCC', 'Validation R2'
]]

# Round numeric values
formatted_results[formatted_results.select_dtypes(include=[np.number]).columns] = \
    formatted_results.select_dtypes(include=[np.number]).round(2)

# Save to CSV
formatted_results.to_csv('model_performance_80_20.csv', index=False)

# Display results
print("\nModel Performance (5-Fold CV on 80% Training, Tested on 20%):\n")
print(formatted_results.to_string(index=False))



Training SVR...
Best parameters for SVR: {'C': 10, 'epsilon': 0.1, 'gamma': 'scale'}

Training Linear Regression...
Best parameters for Linear Regression: {}

Training Ridge...
Best parameters for Ridge: {}

Training Lasso...
Best parameters for Lasso: {}

Training Random Forest...


/tmp/ipython-input-1198175456.py:104: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  train_metrics['pcc'].append(pearsonr(y_tr_orig, y_tr_pred)[0])
/tmp/ipython-input-1198175456.py:110: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  val_metrics['pcc'].append(pearsonr(y_val_orig, y_val_pred)[0])
/tmp/ipython-input-1198175456.py:104: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  train_metrics['pcc'].append(pearsonr(y_tr_orig, y_tr_pred)[0])
/tmp/ipython-input-1198175456.py:110: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  val_metrics['pcc'].append(pearsonr(y_val_orig, y_val_pred)[0])
/tmp/ipython-input-1198175456.py:104: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  train_metrics['pcc'].append(pearsonr(y_tr_orig, y_tr_pred)[0])
/tmp/ipython-in

Streaming output truncated to the last 5000 lines.
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with posit